# GM40.2 — Khảo sát projection GM18 đã bàn giao

Chạy offline từ project; giữ nguyên bundle39 món, không gọi AI/network/DB. Import và HTTP QA có runner riêng.


In [ ]:
from pathlib import Path
import importlib.util, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data-preparation/scripts/catalogue_projection.py').is_file())
spec = importlib.util.spec_from_file_location('gm40_notebook', ROOT/'data-preparation/scripts/catalogue_projection.py')
projection = importlib.util.module_from_spec(spec)
spec.loader.exec_module(projection)
STAGE = ROOT/'data-preparation/datasets/gm40-rich-stage'
print(ROOT, STAGE, sep='\n')


## 1. Dữ liệu nguồn và schema
Không coi [] là phù hợp mọi bữa. Hash và ID được giữ nguyên.


In [ ]:
bundle, source_sha = projection.candidate(ROOT/projection.contract.BUNDLE_PATH, ROOT/projection.contract.MANIFEST_PATH)
display({name: len(rows) for name, rows in bundle['entities'].items()})
print('Dataset', bundle['datasetVersion'], 'Source SHA256', source_sha)
display(bundle['entities']['dishes'][0])


## 2. Card nguồn → storage → API
Projection tái sử dụng importer GM18. Giá chọn observation mới nhất; trùng timestamp hoặc amount null thì không fallback.


In [ ]:
document = projection.document(bundle, source_sha)
print('Release', document['releaseId'])
display(document['cards'][:3])
print('Các trường card:', list(document['cards'][0]))


## 3. Coverage và giá còn thiếu
Cross-tab taxonomy chỉ mô tả, không thêm bộ lọc vào API.


In [ ]:
coverage = projection.coverage(bundle, document)
display(coverage['meals'])
display(coverage['unknown'])
display(coverage['price'])
display(coverage['terms'])


## 4. Schema/template nhận từ GM18
Schema stage thêm metadata projectionVersion, không thay RPC hoặc schema nguồn.


In [ ]:
schema = projection.schema_document()
projection.Draft202012Validator.check_schema(schema)
projection.Draft202012Validator(schema, format_checker=projection.FormatChecker()).validate(document)
display(schema['properties'])


## 5. Stage vào vùng datasets được ignore
Stage chứa bản sao source đầy đủ và card rich. Không import/activate database.


In [ ]:
staged = projection.stage(ROOT/projection.contract.BUNDLE_PATH, STAGE, ROOT/projection.contract.MANIFEST_PATH)
assert staged == document
assert hashlib.sha256((STAGE/'bundle.json').read_bytes()).hexdigest() == source_sha
print('Stage:', STAGE/'stage.json')
print('Source:', STAGE/'bundle.json')


## 6. Kiểm build lặp lại
Artifact được generate có kiểm soát bằng CLI build; notebook không sửa seed nguồn. Chạy QA riêng bằng `python scripts/gm40_rich_projection_check.py --report /tmp/gm40-rich.json`.


In [ ]:
first = projection.build()
assert first == projection.build()
for path, raw in first.items():
    assert path.read_bytes() == raw, path
    print(path.relative_to(ROOT), hashlib.sha256(raw).hexdigest())
